In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../data/product_type_2.csv", header=[0,1])
df

In [ ]:
# 불필요한 id, product_type 컬럼 제거
df = df.drop(columns=[('Process', 'id'), ('Process', 'Product_Type')])

In [ ]:
print("결측치 상위:\n", df.isnull().sum().sort_values(ascending=False).head(10))

print("\n불량 비율:\n", df[('Defect_Flag','Is_Defect')].value_counts(normalize=True))

In [ ]:
y = df[('Defect_Flag','Is_Defect')]

X = df[['Process','Sensor']].copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("train:", X_train.shape)
print("test:", X_test.shape)

print("train 불량률:", y_train.mean())
print("test 불량률:", y_test.mean())

In [ ]:
# 숫자 컬럼만
X_train_num = X_train.select_dtypes(include="number")

In [ ]:
Q1 = X_train_num.quantile(0.25)
Q3 = X_train_num.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

In [ ]:
iqr_summary = pd.DataFrame({
    "이상치 갯수": ((X_train_num < lower) | (X_train_num > upper)).sum()
})

# 이상치 비율 추가
iqr_summary["이상치 비율(%)"] = (
    iqr_summary["이상치 갯수"] / len(X_train_num) * 100
)

# 이상치 있는 컬럼 수
outlier_col_count = (iqr_summary["이상치 갯수"] > 0).sum()
print("이상치가 존재하는 컬럼 수:", outlier_col_count)

In [ ]:
import numpy as np
import pandas as pd

top11_cols = iqr_summary[iqr_summary["이상치 갯수"] > 0].head(11).index

rows = []
for col in top11_cols:
    s = X_train_num[col].dropna()

    lower_val = lower[col]
    upper_val = upper[col]

    mean_val = s.mean()
    min_val  = s.min()
    max_val  = s.max()

    out_mask = (s < lower_val) | (s > upper_val)
    out_cnt = int(out_mask.sum())
    out_rate = out_cnt / len(s) * 100 if len(s) else np.nan

    if mean_val == 0 or np.isclose(mean_val, 0):
        min_pct = np.nan
        max_pct = np.nan
    else:
        min_pct = (min_val - mean_val) / mean_val * 100
        max_pct = (max_val - mean_val) / mean_val * 100

    name = col[1] if isinstance(col, tuple) and len(col) > 1 else str(col)

    rows.append({
        "col": name,
        "최소값": min_val,
        "평균": mean_val,
        "최대값": max_val,
        "min_vs_mean_%": min_pct,
        "max_vs_mean_%": max_pct,
        "outlier_cnt": out_cnt,
        "outlier_rate_%": out_rate,
        "IQR": float(IQR[col])
    })

viz_df = pd.DataFrame(rows).sort_values("outlier_rate_%", ascending=False).reset_index(drop=True)

display(viz_df)

In [ ]:
import matplotlib.pyplot as plt

top8 = iqr_summary.sort_values("이상치 비율(%)", ascending=False).head(8)

labels = [col[1] for col in top8.index]   # MultiIndex에서 컬럼명만 추출
values = top8["이상치 비율(%)"]

plt.figure(figsize=(10,6))

plt.bar(labels, values)

plt.xticks(rotation=45)
plt.ylabel("이상치 비율 (%)")
plt.title("이상치 비율이 높은 컬럼 Top 8")

plt.tight_layout()
plt.show()

In [ ]:
top8_cols = iqr_summary.sort_values("이상치 비율(%)", ascending=False).head(8).index

for col in top11_cols:

    s = X_train_num[col].dropna()

    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False

    plt.figure(figsize=(6,4))

    # x값 점 간격
    x = np.random.normal(0, 0.04, size=len(s))

    plt.scatter(x, s, alpha=0.3, s=8)

    # 평균선 (검정)
    mean_val = s.mean()
    plt.axhline(mean_val, linestyle="--", color="black")

    # 이상치 경계선
    lower_val = lower[col]
    upper_val = upper[col]

    plt.axhline(upper_val, color="red")   # 상한선
    plt.axhline(lower_val, color="blue")  # 하한선

    title_name = col[1] if isinstance(col, tuple) else str(col)

    plt.title(f"{title_name} 이상치 분포도")
    plt.ylabel("Value")
    plt.xticks([])

    plt.tight_layout()
    plt.show()

In [ ]:
df_tmp = pd.DataFrame({
    "Shot": X_train[('Process','Shot')],
    "Defect": y_train
})

# Shot을 10구간으로 나누기
df_tmp["Shot_bin"] = pd.qcut(df_tmp["Shot"], 10)

shot_defect = df_tmp.groupby("Shot_bin")["Defect"].mean()

shot_defect.plot(kind="bar", figsize=(8,4))
plt.xticks(rotation=45)
plt.xlabel("shot 구간")
plt.ylabel("불량주기")
plt.title("shot에 따른 불량주기")
plt.show()